In [ ]:
# 13_rag_chain — gold_core finding -> 근거 검색 -> 근거 기반 권장 조치(citation) -> rag.recommended_action
# retrieval: 12에서 만든 Vector Search index. generation: Databricks Serving endpoint(위젯, 기본 databricks-claude-sonnet-5).
# LLM은 Databricks Serving endpoint 이름만 위젯으로 교체해 선택할 수 있다.
# LLM endpoint가 없거나 근거가 없으면 Gold Core의 deterministic recommended_action으로 fallback 한다(환각 방지).
# MLflow에 finding별 질의/근거/응답 trace를 기록한다.


In [ ]:
# Databricks Runtime 15.4 LTS may not include the Vector Search SDK by default.
# Keep this as a notebook-level guard for manual runs; job tasks also declare the PyPI library.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("databricks.vector_search") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "databricks-vectorsearch"])


In [ ]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("vs_endpoint", "access_drift_vs")
dbutils.widgets.text("index_name", "")
dbutils.widgets.text("llm_endpoint", "databricks-claude-sonnet-5")
dbutils.widgets.text("top_k", "4")
dbutils.widgets.text("limit_findings", "0")
dbutils.widgets.text("llm_timeout_seconds", "60")
dbutils.widgets.text("run_id", "manual")

CATALOG = dbutils.widgets.get("catalog")
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")
INDEX_NAME = dbutils.widgets.get("index_name") or f"{CATALOG}.rag.doc_chunks_index"
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint").strip()
TOP_K = int(dbutils.widgets.get("top_k"))
LIMIT_FINDINGS = int(dbutils.widgets.get("limit_findings"))
LLM_TIMEOUT_SECONDS = int(dbutils.widgets.get("llm_timeout_seconds"))
RUN_ID = dbutils.widgets.get("run_id") or "manual"
TARGET_TABLE = f"{CATALOG}.rag.recommended_action"

print(f"index: {INDEX_NAME} | llm_endpoint: {LLM_ENDPOINT or '(none)'} | top_k: {TOP_K} | llm_timeout={LLM_TIMEOUT_SECONDS}s")


In [ ]:
from databricks.vector_search.client import VectorSearchClient
from mlflow.deployments import get_deploy_client

vsc = VectorSearchClient(disable_notice=True)
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
deploy_client = get_deploy_client("databricks")

# LLM endpoint 가용성 확인. 미등록/미가용이면 이번 run은 deterministic fallback 으로 동작한다.
LLM_AVAILABLE = False
if LLM_ENDPOINT:
    try:
        deploy_client.get_endpoint(LLM_ENDPOINT)
        LLM_AVAILABLE = True
    except Exception as exc:  # noqa: BLE001
        print(f"LLM endpoint '{LLM_ENDPOINT}' 사용 불가 -> deterministic fallback: {exc}")
print(f"LLM_AVAILABLE = {LLM_AVAILABLE}")


In [ ]:
SYSTEM_PROMPT = (
    """너는 보안 조치 보조자다. 아래에 제공된 근거 문서에 있는 내용만으로 권장 조치를 제안한다. 항상 한국어 존댓말로만 답하세요. 문장은 '-합니다', '-하세요', '-됩니다' 체로 작성하세요. '-해야 한다', '-같다', '-하라', 반말/해라체를 사용하지 마세요. 답변은 3~5개의 짧은 bullet로 작성하세요. 각 bullet 끝에는 반드시 근거가 된 [chunk_id] citation을 붙이세요. 제공된 근거로 답할 수 없으면 새로 지어내지 말고 '근거 부족'이라고만 답하세요. 권한을 자동으로 회수하거나 삭제하라고 지시하지 말고, 담당자 확인 후 수행하는 절차로 서술하세요. secret, token, key 원문 값은 답변에 포함하지 마세요."""
)


def build_query(finding: dict) -> str:
    """Risk Card 현재값에서 검색/생성 질의를 만든다."""
    return (
        f"{finding['drift_type']} on {finding['asset_display_name']} "
        f"(sensitivity={finding['asset_sensitivity']}, env={finding['asset_environment']}). "
        f"소유자 {finding['owner_display_name']} 상태 변경, credential active. 권장 조치?"
    )


def principal_kind_of(finding: dict) -> str:
    return "hi" if str(finding.get("subject_type", "")).lower() == "hi" else "nhi"


def retrieve(query: str, finding: dict) -> list:
    res = index.similarity_search(
        query_text=query,
        columns=["chunk_id", "doc_id", "doc_type", "source_title", "source_uri", "chunk_text"],
        filters={"drift_type": finding["drift_type"], "principal_kind": principal_kind_of(finding)},
        num_results=TOP_K,
    )
    return res.get("result", {}).get("data_array") or []


In [ ]:
import json
import urllib.error
import urllib.request


def _workspace_host() -> str:
    # regional/control-plane host(ctx.apiUrl)로 serving invocation을 부르면 404가 난다.
    # workspace URL(spark conf) 우선, 없으면 ctx.apiUrl fallback.
    try:
        return f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"
    except Exception:  # noqa: BLE001
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return ctx.apiUrl().get().rstrip("/")


def llm_payload(messages: list[dict], max_tokens: int = 512) -> dict:
    payload = {"messages": messages, "max_tokens": max_tokens}
    # Databricks Claude/Anthropic endpoints reject temperature, while OpenAI/Llama accept it.
    if "claude" not in LLM_ENDPOINT.lower() and "anthropic" not in LLM_ENDPOINT.lower():
        payload["temperature"] = 0.0
    return payload


def predict_with_timeout(messages: list[dict]) -> dict:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    token = ctx.apiToken().get()
    url = f"{_workspace_host()}/serving-endpoints/{LLM_ENDPOINT}/invocations"
    payload = json.dumps(llm_payload(messages)).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=payload,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=LLM_TIMEOUT_SECONDS) as response:
        return json.loads(response.read().decode("utf-8"))


def generate(finding: dict, query: str, docs: list) -> dict:
    retrieved_ids = [d[0] for d in docs]
    # 근거가 없으면 생성하지 않고 deterministic fallback.
    if not docs:
        return {
            "answer_text": finding["recommended_action"],
            "answer_source": "deterministic_fallback",
            "citations": [],
            "citation_titles": [],
            "retrieved_chunk_ids": retrieved_ids,
            "llm_endpoint": "none",
        }
    # LLM 미가용이면 근거는 보여주되 답변은 deterministic template.
    if not LLM_AVAILABLE:
        return {
            "answer_text": finding["recommended_action"],
            "answer_source": "deterministic_fallback",
            "citations": [],
            "citation_titles": [],
            "retrieved_chunk_ids": retrieved_ids,
            "llm_endpoint": "none",
        }
    # 근거 기반 생성 (citation 강제).
    context = "\n\n".join(f"[{d[0]}] {d[5]}" for d in docs)
    try:
        resp = predict_with_timeout([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"근거:\n{context}\n\n케이스:\n{query}"},
        ])
        answer = resp["choices"][0]["message"]["content"].strip()
    except Exception as exc:  # noqa: BLE001
        # 호출 실패도 환각 대신 fallback.
        return {
            "answer_text": finding["recommended_action"],
            "answer_source": "deterministic_fallback",
            "citations": [],
            "citation_titles": [],
            "retrieved_chunk_ids": retrieved_ids,
            "llm_endpoint": f"error:{type(exc).__name__}",
        }
    # 실제로 인용된 chunk_id만 citation으로 집계.
    cited = [d for d in docs if f"[{d[0]}]" in answer]
    if not cited:
        # 근거를 인용하지 않은 응답은 신뢰하지 않고 fallback (citation 강제).
        return {
            "answer_text": finding["recommended_action"],
            "answer_source": "deterministic_fallback",
            "citations": [],
            "citation_titles": [],
            "retrieved_chunk_ids": retrieved_ids,
            "llm_endpoint": LLM_ENDPOINT,
        }
    return {
        "answer_text": answer,
        "answer_source": "rag",
        "citations": [d[0] for d in cited],
        "citation_titles": [d[3] for d in cited],
        "retrieved_chunk_ids": retrieved_ids,
        "llm_endpoint": LLM_ENDPOINT,
    }


In [ ]:
import mlflow
from datetime import datetime, timezone

findings_df = spark.table(f"{CATALOG}.gold.gold_core")
if LIMIT_FINDINGS > 0:
    findings_df = findings_df.limit(LIMIT_FINDINGS)
findings = [row.asDict() for row in findings_df.collect()]
print(f"findings to process: {len(findings)}")

results = []
now = datetime.now(timezone.utc)

try:
    mlflow.end_run()
except Exception:  # noqa: BLE001
    pass

with mlflow.start_run(run_name=f"rag_chain_{RUN_ID}") as active_run:
    for idx, finding in enumerate(findings, start=1):
        print(f"Processing finding {idx}/{len(findings)}: {finding['risk_case_key']}")
        query = build_query(finding)
        docs = retrieve(query, finding)
        out = generate(finding, query, docs)
        results.append({
            "risk_case_key": finding["risk_case_key"],
            "finding_id": finding["finding_id"],
            "drift_type": finding["drift_type"],
            "query_text": query,
            "answer_text": out["answer_text"],
            "answer_source": out["answer_source"],
            "citations": json.dumps(out["citations"], ensure_ascii=False),
            "citation_titles": json.dumps(out["citation_titles"], ensure_ascii=False),
            "retrieved_chunk_ids": json.dumps(out["retrieved_chunk_ids"], ensure_ascii=False),
            "num_retrieved": len(out["retrieved_chunk_ids"]),
            "llm_endpoint": out["llm_endpoint"],
            "generated_at": now,
            "run_id": RUN_ID,
        })
        try:
            mlflow.log_dict({
                "risk_case_key": finding["risk_case_key"],
                "query": query,
                "retrieved_chunk_ids": out["retrieved_chunk_ids"],
                "answer_source": out["answer_source"],
                "citations": out["citations"],
                "answer_text": out["answer_text"],
            }, f"traces/{finding['risk_case_key']}.json")
        except Exception:  # noqa: BLE001
            pass
    rag_count = sum(1 for r in results if r["answer_source"] == "rag")
    mlflow.log_params({"llm_endpoint": LLM_ENDPOINT or "none", "top_k": TOP_K,
                       "llm_available": LLM_AVAILABLE})
    mlflow.log_metrics({"findings": len(results), "rag_answers": rag_count,
                        "fallback_answers": len(results) - rag_count})
print(f"rag answers: {sum(1 for r in results if r['answer_source'] == 'rag')} / {len(results)}")


In [ ]:
from pyspark.sql import types as T

schema = T.StructType([
    T.StructField("risk_case_key", T.StringType(), False),
    T.StructField("finding_id", T.StringType(), False),
    T.StructField("drift_type", T.StringType(), False),
    T.StructField("query_text", T.StringType(), False),
    T.StructField("answer_text", T.StringType(), False),
    T.StructField("answer_source", T.StringType(), False),
    T.StructField("citations", T.StringType(), False),
    T.StructField("citation_titles", T.StringType(), False),
    T.StructField("retrieved_chunk_ids", T.StringType(), False),
    T.StructField("num_retrieved", T.IntegerType(), False),
    T.StructField("llm_endpoint", T.StringType(), False),
    T.StructField("generated_at", T.TimestampType(), False),
    T.StructField("run_id", T.StringType(), False),
])

assert results, "gold_core finding 이 없습니다. 06_gold 를 먼저 실행하세요."
target_columns = spark.table(TARGET_TABLE).columns
out_df = spark.createDataFrame(results, schema=schema).select(*target_columns)
out_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)

# Smoke: rag 답변은 citation이 반드시 있어야 한다 (citation 강제).
bad = spark.sql(f"""
  SELECT COUNT(*) AS cnt FROM {TARGET_TABLE}
  WHERE answer_source = 'rag' AND (citations IS NULL OR citations = '[]')
""").first()["cnt"]
assert bad == 0, f"citation 없는 rag 답변: {bad}"

written = spark.table(TARGET_TABLE).count()
summary = [
    "rag chain passed",
    f"target: {TARGET_TABLE}",
    f"rows: {written}",
    f"llm_available: {LLM_AVAILABLE}",
]
dbutils.notebook.exit("\n".join(summary))
